In [ ]:
from pymongo import MongoClient
from typing import List, Dict, Any

## Consolidating some MongoDB code queries we used to aggregate our data into new collections and further inspect the data

In [ ]:
# The following queries were used to combine the two datasets on the "parent_asin (shared product ID) key,
# and then filtered to only keep documents with users whose purchases had been verified 

""" 
db["reviews_video_games"].aggregate([
  {
    $lookup: {
      from: "meta_video_games",
      localField: "parent_asin",
      foreignField: "parent_asin",
      as: "meta_data"
    }
  },
  {
    $merge: {
      into: "reviews_with_meta",
      whenMatched: "merge",
      whenNotMatched: "insert"
    }
  }
])

db["reviews_with_meta"].aggregate([
  { $match: { verified_purchase: true } },
  {
    $merge: {
      into: "reviews_with_meta_verified",
      whenMatched: "replace",
      whenNotMatched: "insert"
    }
  }
]) 

"""


## Our initial dataset changed from the first submission, as the original was too large to work with locally. The modeling ideas we'd like to implement will stay the same. Can we build recommender systems for the users based on ratings given (easier) and from their reviews? (more advanced)

## Additionally, we'll implement a DAG that will automatically create aggregated collections from the larger merged dataset (containing product information + reviews.) based on queries.

## Timeline wise, we're stil processing through the data and designing appropriate model(s).

## Some additional exploratory queries:

In [ ]:
# ratings count breakdown
""" db.reviews_with_meta_verified.aggregate([
  {
    $group: {
      _id: "$rating",
      n: { $sum: 1 }
    }
  },
  { $sort: { _id: 1 } }
]) """
# Result
""" {
  _id: 1,
  n: 243551
}
{
  _id: 2,
  n: 114555
}
{
  _id: 3,
  n: 172335
}
{
  _id: 4,
  n: 308642
}

{
  _id: 5,
  n: 1556719
} """

# Lots of 5 star reviews!


In [ ]:
## Taking advantage of "helpful_vote" field
## Does it relate to higher or lower ratings?

""" db.reviews_with_meta_verified.aggregate([
  {
    $group: {
      _id: "$rating",
      avg_helpful: { $avg: "$helpful_vote" },
      total_helpful: { $sum: "$helpful_vote" },
      n: { $sum: 1 }
    }
  },
  { $sort: { _id: 1 } }
]) """


""" {_id: 1,
  avg_helpful: 1.9059539891028983,
  total_helpful: 464197,
  n: 243551
}
{
  _id: 2,
  avg_helpful: 1.2970538169438262,
  total_helpful: 148584,
  n: 114555
}

{  _id: 3,
  avg_helpful: 1.1492674152087503,
  total_helpful: 198059,
  n: 172335
}

{  _id: 4,
  avg_helpful: 1.0166503586679714,
  total_helpful: 313781,
  n: 308642
}

{
  _id: 5,
  avg_helpful: 0.7337387158504521,
  total_helpful: 1142225,
  n: 1556719
} """

# Looks like people find the 1 star and 5 star reviews the most helpful!
